# SelTDA VQAScore scoring on Kaggle

Notebook chấm từng cặp `(image, question + answer)` bằng `clip-flant5-xl`, ghi vào `record['scores']['vqascore']`, và xuất cache tương thích `filter_pseudo.py`.

Trên Kaggle: bật **GPU** và **Internet** (lần đầu cần tải package/model), attach dataset chứa JSON và ảnh COCO, rồi sửa path ở cell cấu hình. Cell cài dependency sẽ tự restart kernel một lần; sau khi Kaggle reconnect, chạy lại notebook. Output nằm trong `/kaggle/working`.

In [ ]:
import importlib.metadata as metadata
import os, subprocess, sys

required = {
    'numpy': '1.26.4',
    'scipy': '1.11.4',
    'transformers': '4.36.1',
    'diffusers': '0.29.2',
    't2v-metrics': '1.2',
}

def installed_version(package):
    try:
        return metadata.version(package)
    except metadata.PackageNotFoundError:
        return None

needs_install = any(installed_version(name) != version for name, version in required.items())
needs_install = needs_install or installed_version('clip') is None
if needs_install:
    specs = [f'{name}=={version}' for name, version in required.items()]
    specs.append('git+https://github.com/openai/CLIP.git')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--force-reinstall', *specs])
    print('Đã cài dependency. Kaggle đang restart kernel; sau khi reconnect hãy Run All lại.', flush=True)
    os._exit(0)

print({name: installed_version(name) for name in required})

In [ ]:
from pathlib import Path

# Sửa hai path đầu theo tên Kaggle Dataset đã attach.
INPUT_JSON = Path('/kaggle/input/datasets/phong2004/seltda/synthetic_data_raw.json')
IMAGE_ROOT = Path('/kaggle/input/datasets/mathew0george/coco-2017-unlabeled')
RESUME_JSON = None  # Path output của session trước, nếu có

OUTPUT_JSON = Path('/kaggle/working/synthetic_data_vqascore_scored.json')
SCORE_CACHE_DIR = Path('/kaggle/working/score_cache')
CACHE_JSON = SCORE_CACHE_DIR / f'{INPUT_JSON.stem}__vqascore.json'
MODEL_NAME = 'clip-flant5-xl'
BATCH_SIZE = 8       # Giảm còn 4 hoặc 2 nếu hết VRAM
SAVE_EVERY = 100     # Lưu sau mỗi 100 batch
LIMIT = None         # Đặt 32 để smoke test

assert INPUT_JSON.is_file(), f'Không thấy input: {INPUT_JSON}'
assert IMAGE_ROOT.is_dir(), f'Không thấy ảnh: {IMAGE_ROOT}'

In [ ]:
import json, os
import torch
from tqdm.auto import tqdm

assert torch.cuda.is_available(), 'Bật GPU trong Settings > Accelerator'
DEVICE = 'cuda'
print('GPU:', torch.cuda.get_device_name(0))

with INPUT_JSON.open(encoding='utf-8') as f:
    input_records = json.load(f)
assert isinstance(input_records, list), 'JSON phải là list record'

resume_path = Path(RESUME_JSON) if RESUME_JSON else OUTPUT_JSON
if resume_path.is_file():
    with resume_path.open(encoding='utf-8') as f:
        records = json.load(f)
    assert len(records) == len(input_records), 'Resume không khớp số record'
    for old, new in zip(records, input_records):
        assert (old.get('image'), old.get('question')) == (new.get('image'), new.get('question')), 'Resume không cùng dataset/order'
else:
    records = input_records

work_records = records if LIMIT is None else records[:LIMIT]
pending = [i for i, r in enumerate(work_records) if 'vqascore' not in (r.get('scores') or {})]
print(f'Tổng {len(work_records):,}; đã có {len(work_records)-len(pending):,}; còn {len(pending):,}')

In [ ]:
# Hỗ trợ cả layout ảnh phẳng của SelTDA và train2017/val2017 trên Kaggle.
image_index = None

def resolve_image(image_field):
    global image_index
    direct = IMAGE_ROOT / image_field
    if direct.is_file():
        return direct
    if image_index is None:
        print('Đang lập index ảnh đệ quy...')
        image_index = {}
        for p in IMAGE_ROOT.rglob('*'):
            if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}:
                image_index.setdefault(p.name, p)
    result = image_index.get(Path(image_field).name)
    if result is None:
        raise FileNotFoundError(image_field)
    return result

def qa_text(record):
    answer = record['answer']
    if isinstance(answer, list):
        answer = answer[0] if answer else ''
    return f"{record['question']} {answer}".strip()

for i in pending[:20]:
    resolve_image(work_records[i]['image'])
print('Path ảnh: OK')

In [ ]:
import t2v_metrics

scorer = t2v_metrics.VQAScore(model=MODEL_NAME, device=DEVICE)
print('Loaded:', MODEL_NAME)

In [ ]:
def paired_values(raw, n):
    value = raw.detach().float().cpu() if isinstance(raw, torch.Tensor) else torch.as_tensor(raw, dtype=torch.float32)
    if value.ndim == 2 and value.shape == (n, n):
        value = value.diagonal()
    else:
        value = value.reshape(-1)
    if value.numel() != n:
        raise ValueError(f'Model trả shape {tuple(value.shape)}, cần {n} scores')
    return value.tolist()

def score_batch(paths, texts):
    # forward: n paired scores; __call__ fallback: ma trận n x n rồi lấy diagonal.
    model = getattr(scorer, 'model', None)
    with torch.inference_mode():
        raw = model.forward(paths, texts) if model is not None and hasattr(model, 'forward') else scorer(images=paths, texts=texts)
    return paired_values(raw, len(paths))

def score_with_oom_split(indices):
    try:
        paths = [str(resolve_image(work_records[i]['image'])) for i in indices]
        texts = [qa_text(work_records[i]) for i in indices]
        return score_batch(paths, texts)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(indices) == 1:
            raise
        mid = len(indices) // 2
        return score_with_oom_split(indices[:mid]) + score_with_oom_split(indices[mid:])

def atomic_json_dump(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with tmp.open('w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False)
    os.replace(tmp, path)

batches = [pending[i:i+BATCH_SIZE] for i in range(0, len(pending), BATCH_SIZE)]
for batch_no, indices in enumerate(tqdm(batches, desc='VQAScore'), start=1):
    for i, score in zip(indices, score_with_oom_split(indices)):
        if work_records[i].get('scores') is None:
            work_records[i]['scores'] = {}
        work_records[i]['scores']['vqascore'] = float(score)
    if batch_no % SAVE_EVERY == 0:
        atomic_json_dump(records, OUTPUT_JSON)
atomic_json_dump(records, OUTPUT_JSON)
print('Scored records:', OUTPUT_JSON)

In [ ]:
# Cache version 2 tương thích filtering.score_cache.
def record_key(record):
    return f"{record.get('image', '')}\0{record.get('question', '')}"

cache_scores = {record_key(r): float(r['scores']['vqascore']) for r in work_records if 'vqascore' in (r.get('scores') or {})}
cache = {'version': 2, 'input': str(INPUT_JSON), 'n_records': len(records), 'gate': 'vqascore', 'scores': cache_scores}
atomic_json_dump(cache, CACHE_JSON)
values = list(cache_scores.values())
print('Cache:', CACHE_JSON)
if values:
    print(f'n={len(values):,}, min={min(values):.6f}, mean={sum(values)/len(values):.6f}, max={max(values):.6f}')